# EEG Oddball: kiểm tra dữ liệu → tiền xử lý → từng mô hình → so sánh

Notebook chỉ điều phối các module trong repo. Nhãn: **standard/frequent = 0**, **target = 1**; loại distractor. Chọn Colab **CPU** hoặc **GPU NVIDIA**. GPU được thử cho SVM và Logistic Regression; các bước còn lại giữ CPU. Mỗi lệnh lỗi sẽ dừng cell bằng exception.

Chạy tuần tự, xem QC trước khi tiếp tục. Kết quả nằm trong `artifacts/` liên kết với Drive. Đây là đánh giá offline trên người chưa có trong tập train; không phải đánh giá trực tuyến.

## 1. Cài đặt, Google Drive và ghi nhận môi trường

Không thay thế thư mục kết quả hiện có. Repo đang có thay đổi chưa commit sẽ được giữ lại và cell báo lỗi để bạn tự kiểm tra. Cài GPU là tùy chọn: nếu không tương thích, `auto` ghi lý do rồi dùng CPU.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
REPO = Path('/content/Miniproject-Computational-NeuroScience')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/namkhanhdoan2305/Miniproject-Computational-NeuroScience.git', str(REPO)], check=True)
else:
    changes = subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True)
    if changes.strip():
        raise RuntimeError('Repo có thay đổi chưa commit; hãy lưu lại trước khi cập nhật.')
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

ENABLE_GPU = True
has_nvidia = shutil.which('nvidia-smi') is not None
if ENABLE_GPU and has_nvidia:
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-gpu.txt'], check=True)
    except subprocess.CalledProcessError:
        print('Không cài được cuML/CUDA 12. Chế độ auto sẽ thử CPU và ghi lý do; kiểm tra log cài đặt ở trên.')

DRIVE_OUTPUT = Path('/content/drive/MyDrive/oddball_artifacts')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
ART = REPO / 'artifacts'
if not ART.exists() and not ART.is_symlink():
    ART.symlink_to(DRIVE_OUTPUT, target_is_directory=True)
if not ART.is_symlink() or ART.resolve() != DRIVE_OUTPUT.resolve():
    raise RuntimeError('artifacts chưa trỏ đúng Drive. Hãy sao lưu/di chuyển thư mục cũ trước; cell không tự xóa dữ liệu.')

from src.common import load_config, package_versions, write_json
from src.backends import gpu_status
from src.notebook import run_step, show_model, show_images, show_qc
CONFIG = load_config()
revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
environment = {'git_commit': revision, 'python': sys.version, 'versions': package_versions(), 'gpu': gpu_status()}
write_json(ART / 'environment.json', environment)
print(json.dumps(environment, indent=2))


## 2. Dry-run sub-01 và xem tín hiệu thật trước/sau xử lý

Khi chưa có cohort, xử lý sub-01 và giữ checkpoint để dùng lại trong bước đầy đủ. Nếu đã có cohort trên Drive, chỉ tải lại sub-01 **khi thiếu QC**, không ghi đè `X.npy` hay đặc trưng của 42 người.

QC gồm số sự kiện giữ/loại, trace trước lọc/sau lọc/sau ICA, PSD, tương quan và bản đồ ICA, ERP trên **cùng trial**, heatmap từng trial và bản đồ P300 da đầu. `raw` trong hình là tín hiệu trước bộ lọc, sau chọn kênh và nội suy kênh đã được BIDS đánh dấu. Kiểm tra cả thành phần ICA khi dùng Fp1/Fp2 làm EOG thay thế; tương quan cao không tự chứng minh thành phần đó chỉ chứa nhiễu.

In [ ]:
from src.common import read_json
meta_path = ART / 'metadata.json'
if not meta_path.exists() or read_json(meta_path)['dry_run']:
    run_step('src.preprocess', '--dry-run', '--resume')
elif not list((ART / 'qc' / 'sub-01').glob('*/metadata.json')):
    run_step('src.preprocess', '--qc-only', '--subjects', 'sub-01', '--revision', read_json(meta_path)['revision'])
else:
    print('Dùng QC đã có; không tải lại dữ liệu thô.')
show_qc(ART, 'sub-01')


## 3. Xử lý đầy đủ và xem phân bố dữ liệu

Mỗi lần chỉ tải một người qua `snapshot_download`, xử lý và xóa raw trong `finally`. Đã có đủ 42 người cùng cấu hình thì dùng lại arrays. `--resume` dùng checkpoint nếu lần trước bị ngắt.

Biểu đồ tổng quan dùng trial thực tế sau loại distractor và nhiễu; kiểm tra tỷ lệ target của từng người. Grand-average ERP có CI 95% **theo người**, không tính các trial như quan sát độc lập.

In [ ]:
from src.common import load_epochs
cohort_ready = False
if meta_path.exists():
    meta = read_json(meta_path)
    same_settings = (meta['config']['preprocessing'] == CONFIG['preprocessing']
                     and meta['config']['events'] == CONFIG['events']
                     and meta['config']['seed'] == CONFIG['seed'])
    cohort_ready = (not meta['dry_run'] and len(meta['subjects']) == CONFIG['dataset']['expected_subjects'] and same_settings)
if not cohort_ready:
    run_step('src.preprocess', '--resume')
else:
    print('Dùng lại cohort hoàn chỉnh trên Drive:', len(meta['subjects']), 'người')
load_epochs(ART)  # kiểm tra schema trước bước tiếp theo
run_step('src.visualize', '--data-only')
import pandas as pd
from IPython.display import display
print('Tổng số trial theo lớp:')
display(pd.read_csv(ART / 'dataset_overview.csv'))
show_images(ART, ['dataset_overview.png', 'processed_trial_examples.png', 'fig1_grand_average_erp.png'])


## 4. Trích xuất đặc trưng

P300 mean/peak/latency/diện tích 300–600 ms tại từng kênh và ROI Fz/Cz/Pz, cùng band power, entropy và thống kê wavelet. Mặc định **30 × 26 + 5 = 785 đặc trưng**. Mean/AUC có dấu giữ thông tin cực tính; peak/latency single-trial có thể nhạy với nhiễu. Band power không thay thế hình thái ERP, và delta trên epoch một giây có độ phân giải thấp.

Đặc trưng được tạo trên từng trial; scaler chỉ fit trong train fold. Bước này giữ schema baseline hiện tại để dùng lại dữ liệu trên Drive. Notebook không tự thêm xDAWN, chuẩn hóa toàn subject mới hoặc chọn ngưỡng theo outer test.

In [ ]:
from src.features import load_features
if (ART / 'feature_metadata.json').exists():
    _, _, _, feature_names, feature_info = load_features(ART)
    if feature_info['settings'] != CONFIG['features']:
        raise RuntimeError('Cấu hình đặc trưng thay đổi. Chạy lại src.features chủ động trước khi train.')
    print('Dùng lại', len(feature_names), 'đặc trưng có cùng schema/dataset.')
else:
    run_step('src.features')
from src.visualize import plot_distributions
plot_distributions(ART, CONFIG['visualization'])
show_images(ART, ['feature_distributions.png'])


## 5. Shrinkage LDA — baseline ERP tuyến tính

LDA mô hình hóa phân bố từng lớp với hiệp phương sai chung; shrinkage ổn định ước lượng khi nhiều đặc trưng tương quan. Priors `[0.5, 0.5]` tăng sự cân bằng giữa hai lớp. Không có chuỗi epoch-loss như mạng nơ-ron; hình learning curve so sánh train/validation khi tăng **số người train**, trên một inner split cố định.

In [ ]:
run_step('src.train_lda', '--device', 'auto', '--resume')
show_model('lda', ART)


## 6. SVM RBF — kiểm tra ranh giới phi tuyến

RBF cho phép ranh giới cong trong không gian đặc trưng; `class_weight='balanced'` tăng chi phí bỏ sót target hiếm. SVM dùng decision score để tính ROC/AP; không gọi score này là xác suất. Chế độ `auto` thử cuML GPU khi có, nếu không sẽ dùng sklearn CPU và ghi lý do. Learning curve không phải loss theo vòng tối ưu.

In [ ]:
run_step('src.train_svm', '--device', 'auto', '--resume')
show_model('svm', ART)


## 7. Random Forest — tương tác đặc trưng và mức quan trọng

Rừng cây đánh giá quan hệ phi tuyến giữa các thống kê tín hiệu, dùng trọng số lớp cân bằng. Giữ sklearn CPU để bảo toàn cấu hình này. Biểu đồ log loss theo **số cây** dùng cùng inner train/validation subjects; thêm cây không phải gradient descent và không đảm bảo validation loss luôn giảm. Không dùng OOB theo trial để giả làm đánh giá người mới.

Top feature importance lấy từ các fold train; chỉ mô tả mô hình, không chứng minh kênh não có tác dụng nhân quả.

In [ ]:
run_step('src.train_random_forest', '--device', 'auto', '--resume')
show_model('random_forest', ART)


## 8. Logistic Regression L2 — baseline tuyến tính phân biệt

Logistic Regression tối ưu xác suất phân lớp với regularization L2 và balanced weights. So sánh với LDA để kiểm tra giả định mô hình sinh/phân biệt. GPU được thử khi khả dụng.

Biểu đồ iteration-loss là **lượt chẩn đoán CPU riêng** dùng callback L-BFGS, trên inner subjects; không phải lịch sử của GPU hay outer CV. Hình phân biệt objective có trọng số + L2 và log loss không trọng số. Validation loss có thể tăng, và không được dùng để điều chỉnh benchmark. Các lỗi hội tụ của benchmark được thử tăng giới hạn một lần; nếu vẫn không hội tụ thì báo lỗi và không đánh dấu hoàn tất.

In [ ]:
run_step('src.train_logistic_regression', '--device', 'auto', '--resume')
show_model('logistic_regression', ART)


## 9. So sánh bốn mô hình trên cùng người kiểm tra

Bước này **không train lại**; chỉ chấp nhận bốn run hoàn chỉnh có cùng dataset, đặc trưng, tham số và GroupKFold. Nếu đổi cấu hình hoặc backend, chạy lại mô hình tương ứng với `--overwrite` thay cho `--resume`.

Đọc F1 target (tiêu chí so sánh định trước), Balanced Accuracy, Recall/Precision, ROC-AUC và AP; dùng confusion matrix để xem false negative. Accuracy dễ cao do đa số là standard. SD giữa các fold là mô tả độ biến thiên, không phải CI hay kiểm định khác biệt. “Đứng đầu” trên các fold này chưa chứng minh mô hình tốt hơn trên quần thể mới.

In [ ]:
run_step('src.compare_ml')
results = pd.read_csv(ART / 'ml_benchmark_results.csv')
columns = ['model', 'backend', 'accuracy_mean', 'balanced_accuracy_mean', 'precision_mean', 'recall_mean', 'f1_mean', 'f1_std', 'roc_auc_mean', 'average_precision_mean', 'fit_seconds_mean']
display(results[columns].sort_values('f1_mean', ascending=False))
show_images(ART, ['model_comparison.png', 'fig2_roc_and_pr_curves.png', 'fig3_confusion_matrices.png', 'fig4_feature_importance.png'])
report = read_json(ART / 'ml_benchmark_results.json')
print('Đứng đầu theo từng chỉ số (thăm dò):', report['best_by_metric'])
print('Accuracy của baseline luôn chọn lớp đa số:', report['majority_accuracy_baseline'])
print('Toàn bộ metrics, mô hình, hình và QC:', ART.resolve())
